#### Graph Convolutional Network (GCN)

In [3]:
import dgl
import dgl.function as fn
import time
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
from dgl import DGLGraph
from pytorchtools import EarlyStopping # git clone https://github.com/Bjarten/early-stopping-pytorch
from dgl.data import citation_graph as citegrh
import networkx as nx


gcn_msg = fn.copy_src(src='h', out='m')
gcn_reduce = fn.sum(msg='m', out='h')

Using backend: pytorch


Layers and model definition

In [4]:
class GCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_feats, out_feats)
        
    def forward(self, g, feature):
        with g.local_scope():
            g.ndata['h'] = feature
            g.update_all(gcn_msg, gcn_reduce)
            h = g.ndata['h']
            return self.linear(h)

class Net(nn.Module):
    def __init__(self, in_feats, out_feats):
        super(Net, self).__init__()
        self.layer1 = GCNLayer(in_feats, 16)
        self.layer2 = GCNLayer(16, out_feats)
        
    def forward(self, g, features):
        x = F.relu(self.layer1(g, features))
        x = self.layer2(g, x)
        return x

Load datasets

In [5]:
def load_cora_data():
    data = citegrh.load_cora()
    features = torch.FloatTensor(data.features)
    labels = torch.LongTensor(data.labels)
    train_mask = torch.BoolTensor(data.train_mask)
    test_mask = torch.BoolTensor(data.test_mask)
    g = DGLGraph(data.graph)
    return g, features, labels, train_mask, test_mask

def load_citeseer_data():
    data = citegrh.load_citeseer()
    features = torch.FloatTensor(data.features)
    labels = torch.LongTensor(data.labels)
    train_mask = torch.BoolTensor(data.train_mask)
    test_mask = torch.BoolTensor(data.test_mask)
    g = DGLGraph(data.graph)
    return g, features, labels, train_mask, test_mask

def load_pubmed_data():
    data = citegrh.load_pubmed()
    features = torch.FloatTensor(data.features)
    labels = torch.LongTensor(data.labels)
    train_mask = torch.BoolTensor(data.train_mask)
    test_mask = torch.BoolTensor(data.test_mask)
    g = DGLGraph(data.graph)
    return g, features, labels, train_mask, test_mask

evaluation & seed set

In [6]:
def evaluate(model, g, features, labels, mask):
    model.eval()
    with torch.no_grad():
        logits = model(g, features)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = torch.max(logits, dim=1)
        correct = torch.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)
    
def set_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

pass args to dataset's load func

In [7]:
g, features, labels, train_mask, test_mask = load_cora_data()
path_cora = "/home/matalan/venv/graphlex_beta/early-stopping-pytorch/checkpoints/cora/"

g, features, labels, train_mask, test_mask = load_citeseer_data()
path_citeseer = "/home/matalan/venv/graphlex_beta/early-stopping-pytorch/checkpoints/citeseer/"

g, features, labels, train_mask, test_mask = load_pubmed_data()
path_pubmed = "/home/matalan/venv/graphlex_beta/early-stopping-pytorch/checkpoints/pubmed/"

/home/matalan/.local/lib/python3.8/site-packages/dgl/data/citation_graph.py:140: RuntimeWarning: divide by zero encountered in power
  r_inv = np.power(rowsum, -1).flatten()


Finished data loading and preprocessing.
  NumNodes: 3327
  NumEdges: 9228
  NumFeats: 3703
  NumClasses: 6
  NumTrainingSamples: 120
  NumValidationSamples: 500
  NumTestSamples: 1000
Finished data loading and preprocessing.
  NumNodes: 19717
  NumEdges: 88651
  NumFeats: 500
  NumClasses: 3
  NumTrainingSamples: 60
  NumValidationSamples: 500
  NumTestSamples: 1000


training

In [8]:
def train_model(model, optimizer, patience, n_epochs):

    dur =[]
    train_losses = []
    valid_losses = []
    avg_train_losses = []
    avg_valid_losses = [] 
    
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    for epoch in range(n_epochs):
        if epoch >=3:
            t0 = time.time()
        
        model.train()
        logits = model(g, features)
        logp = F.log_softmax(logits, 1)
        loss = F.nll_loss(logp[train_mask], labels[train_mask])
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch >=3:
            dur.append(time.time() - t0)
            
        acc = evaluate(model, g, features, labels, test_mask) 
        
        print("Epoch {:05d} | loss {:.4f} | Test Acc {:.4f} | Time(s) {:.4f}".format(
        epoch, loss.item(), acc, np.mean(dur)))
        
        model.eval()
        with torch.no_grad():
            logits = model(g, features)
            logp = F.log_softmax(logits, 1)
            loss_val = F.nll_loss(logp[test_mask], labels[test_mask])
            valid_losses.append(loss_val.item())
        
        valid_loss = np.average(valid_losses)
        epoch_len = len(str(n_epochs))
        
        train_losses = []
        valid_losses = []
        
        early_stopping(valid_loss, model)
        
        if early_stopping.early_stop:
            print("Early stopping")
            break
    
    return  model, avg_train_losses, avg_valid_losses

save all models

In [13]:
def save_all_models(n_models, path):
    path = path
    for i in range(n_models):
        set_seed(42+i)
        net = Net(in_feats = features.size()[1],
                  out_feats = len(labels.unique()))
        optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)
        model, avg_train_losses, avg_valid_losses = train_model(net, optimizer, 10, 50)
        filename = path + 'checkpoint_' + str(i + 1) + '.pth'
        torch.save(net.state_dict(), filename)
        #print(net.parameters())
        feature_extractor = torch.nn.Sequential(*list(net.children())[:-1])
    return

In [14]:
save_all_models(1, path_cora)

#save_all_models(10, path_citeseer)
#save_all_models(10, path_pubmed)

Epoch 00000 | loss 1.1119 | Test Acc 0.3290 | Time(s) nan
Validation loss decreased (inf --> 1.062913).  Saving model ...
Epoch 00001 | loss 1.0375 | Test Acc 0.4460 | Time(s) nan
Validation loss decreased (1.062913 --> 1.035090).  Saving model ...
Epoch 00002 | loss 0.9301 | Test Acc 0.5440 | Time(s) nan
Validation loss decreased (1.035090 --> 1.005870).  Saving model ...
Epoch 00003 | loss 0.8505 | Test Acc 0.6330 | Time(s) 0.1113
Validation loss decreased (1.005870 --> 0.938385).  Saving model ...
Epoch 00004 | loss 0.7560 | Test Acc 0.6910 | Time(s) 0.1175
Validation loss decreased (0.938385 --> 0.905758).  Saving model ...
Epoch 00005 | loss 0.6785 | Test Acc 0.6980 | Time(s) 0.1271
Validation loss decreased (0.905758 --> 0.895973).  Saving model ...
Epoch 00006 | loss 0.6052 | Test Acc 0.6950 | Time(s) 0.1197
Validation loss decreased (0.895973 --> 0.892002).  Saving model ...
Epoch 00007 | loss 0.5481 | Test Acc 0.7040 | Time(s) 0.1182
Validation loss decreased (0.892002 --> 0.8